In [1]:
from pathlib import Path
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Trỏ thẳng vào thư mục data bên trong project
BASE_DATA_DIR = PROJECT_ROOT / "data" 

# --- CHỌN BỘ DỮ LIỆU ĐỂ TRAIN TẠI ĐÂY -

DATA_DIR = BASE_DATA_DIR / "Cow_data"

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print(f"Đang nạp dữ liệu từ: {DATA_DIR.name}")
print("Thư mục Data tồn tại:", DATA_DIR.exists())
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

TRAIN_MODE = True

Project root: d:\NCKH\livestock-diseases-ai
Đang nạp dữ liệu từ: Cow_data
Thư mục Data tồn tại: True
Torch: 2.5.1+cu121
Device: cuda


In [2]:
from src.utils import set_seed
from src.dataset import load_dataset, make_loaders
from src.model import build_model, count_parameters, unfreeze_backbone, load_checkpoint
from src.train import train_model
from src.evaluate import run_full_evaluation, plot_training_curves

set_seed(42)
print("Project modules imported successfully.")

Project modules imported successfully.


In [3]:
# Tự động đọc data và weights từ dataset.py mới
train_dataset, val_dataset, test_dataset, info = load_dataset(DATA_DIR)

train_loader, val_loader, test_loader = make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32, # Giữ ở mức 32 để tránh tràn RAM
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

print("Dataset loaded successfully.")
print("Split sizes:", info["split_sizes"])
print("Number of classes:", info["n_classes"])
print("Trọng số phân lớp (Class Weights):", info["class_weights"])

Dataset loaded successfully.
Split sizes: {'train': 2233, 'val': 477, 'test': 484, 'total': 3194}
Number of classes: 4
Trọng số phân lớp (Class Weights): tensor([1.1347, 0.7130, 0.6900, 3.7466])


In [4]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Min pixel value:", images.min().item())
print("Max pixel value:", images.max().item())
print("First 10 labels:", labels[:10].tolist())

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Min pixel value: -2.1179039478302
Max pixel value: 2.640000104904175
First 10 labels: [2, 1, 1, 2, 2, 0, 2, 1, 0, 1]


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tự động scale số lớp theo n_classes của dataset
model = build_model(
    n_classes=info["n_classes"],
    freeze_backbone=True,
).to(device)

params = count_parameters(model)

print("Device:", device)
print("Model device:", next(model.parameters()).device)
print("Model created successfully.")
print("Parameters:", params)

Device: cuda
Model device: cuda:0
Model created successfully.
Parameters: {'total': 11178564, 'trainable': 2052, 'frozen': 11176512}


In [6]:
with torch.no_grad():
    sample_outputs = model(images.to(device))

print("Sample output shape:", sample_outputs.shape)

Sample output shape: torch.Size([32, 4])


In [7]:
phase1_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=10,
    learning_rate=0.001,
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_cow_phase1_best.pth"),
    history_path=str(RESULTS_DIR / "history_cow_phase1.json"),
    phase_name="phase1_frozen_backbone",
)

print("Phase 1 training completed.")


Training phase1_frozen_backbone
Epochs: 10
Learning rate: 0.001
Trainable parameters: 2,052


Epoch 01/10 | Train Loss: 1.6224 | Train Acc: 0.3995 | Val Loss: 0.8677 | Val Acc: 0.6436 | Time: 29s
  --> Best checkpoint saved! (Val Acc: 0.6436)


Epoch 02/10 | Train Loss: 1.0661 | Train Acc: 0.5773 | Val Loss: 0.7285 | Val Acc: 0.7065 | Time: 18s
  --> Best checkpoint saved! (Val Acc: 0.7065)


Epoch 03/10 | Train Loss: 0.8814 | Train Acc: 0.6413 | Val Loss: 0.6338 | Val Acc: 0.7400 | Time: 16s
  --> Best checkpoint saved! (Val Acc: 0.7400)


Epoch 04/10 | Train Loss: 0.8422 | Train Acc: 0.6820 | Val Loss: 0.6298 | Val Acc: 0.7400 | Time: 16s


...

Epoch 06/10 | Train Loss: 0.7093 | Train Acc: 0.7241 | Val Loss: 0.5406 | Val Acc: 0.7904 | Time: 9s
  --> Best checkpoint saved! (Val Acc: 0.7904)


Epoch 07/10 | Train Loss: 0.6943 | Train Acc: 0.7389 | Val Loss: 0.5466 | Val Acc: 0.7694 | Time: 10s


Epoch 08/10 | Train Loss: 0.6422 | Train Acc: 0.7385 | Val Loss: 0.5392 | Val Acc: 0.7862 | Time: 8s


Epoch 09/10 | Train Loss: 0.6398 | Train Acc: 0.7474 | Val Loss: 0.5677 | Val Acc: 0.7547 | Time: 11s


Epoch 10/10 | Train Loss: 0.6397 | Train Acc: 0.7492 | Val Loss: 0.5493 | Val Acc: 0.7757 | Time: 13s
Phase 1 training completed.


In [8]:
phase1_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase1",
    prefix="cow_",
)

plot_training_curves(
    phase1_history,
    save_path=FIGURES_DIR / "training2_curves_phase1.png",
    prefix="cow_",
)
print("Phase 1 evaluation and plotting completed.")


--- Evaluation Results (phase1) ---
Accuracy   : 0.7934
Macro F1   : 0.7881
Weighted F1: 0.7945
Phase 1 evaluation and plotting completed.


In [9]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_cow_phase1_best.pth"),
)

unfreeze_backbone(model)

params = count_parameters(model)

print("Best Phase 1 checkpoint loaded.")
print("Backbone unfrozen for Phase 2.")
print("Parameters:", params)

Best Phase 1 checkpoint loaded.
Backbone unfrozen for Phase 2.
Parameters: {'total': 11178564, 'trainable': 11178564, 'frozen': 0}


d:\NCKH\livestock-diseases-ai\src\model.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


In [10]:
phase2_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=15,
    learning_rate=0.0001, # LR nhỏ để tinh chỉnh mượt mà
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_cow_phase2_best.pth"),
    history_path=str(RESULTS_DIR / "history_cow_phase2.json"),
    phase_name="phase2_full_finetuning",
)

print("Phase 2 training completed.")


Training phase2_full_finetuning
Epochs: 15
Learning rate: 0.0001
Trainable parameters: 11,178,564


Epoch 01/15 | Train Loss: 0.4505 | Train Acc: 0.8294 | Val Loss: 0.3446 | Val Acc: 0.8742 | Time: 16s
  --> Best checkpoint saved! (Val Acc: 0.8742)


Epoch 02/15 | Train Loss: 0.2484 | Train Acc: 0.9055 | Val Loss: 0.2694 | Val Acc: 0.9182 | Time: 17s
  --> Best checkpoint saved! (Val Acc: 0.9182)


Epoch 03/15 | Train Loss: 0.1677 | Train Acc: 0.9369 | Val Loss: 0.3124 | Val Acc: 0.8910 | Time: 16s


Epoch 04/15 | Train Loss: 0.1121 | Train Acc: 0.9512 | Val Loss: 0.2606 | Val Acc: 0.9036 | Time: 15s


...

Epoch 11/15 | Train Loss: 0.0338 | Train Acc: 0.9848 | Val Loss: 0.2235 | Val Acc: 0.9413 | Time: 15s
  --> Best checkpoint saved! (Val Acc: 0.9413)


Epoch 12/15 | Train Loss: 0.0201 | Train Acc: 0.9906 | Val Loss: 0.2049 | Val Acc: 0.9476 | Time: 15s
  --> Best checkpoint saved! (Val Acc: 0.9476)


Epoch 13/15 | Train Loss: 0.0287 | Train Acc: 0.9893 | Val Loss: 0.2125 | Val Acc: 0.9287 | Time: 16s


Epoch 14/15 | Train Loss: 0.0267 | Train Acc: 0.9915 | Val Loss: 0.2108 | Val Acc: 0.9371 | Time: 15s


Epoch 15/15 | Train Loss: 0.0174 | Train Acc: 0.9924 | Val Loss: 0.2132 | Val Acc: 0.9392 | Time: 15s
Phase 2 training completed.


In [12]:
phase2_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase2",
    prefix="cow_",
)

plot_training_curves(
    phase2_history,
    save_path=FIGURES_DIR / "training2_curves_phase2.png",
    prefix="cow_",
)
print("Phase 2 evaluation completed.")


--- Evaluation Results (phase2) ---
Accuracy   : 0.9091
Macro F1   : 0.9064
Weighted F1: 0.9092
Phase 2 evaluation completed.
